# 10 — Typing, Contracts, and Protocols

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/10', {1: 'ae89c0b74cf82838', 2: '51b297b0bb1c74c8', 3: 'b700d9301db2e3a8', 4: 'eb9fe5c674a5a4c2', 5: 'ae6a2a6a52d9278f', 6: '96316f742cf2b919', 7: '16b14092acb890e2', 8: 'c8e0c8d696f7e09e'}, practice='intermediate_03_data_quality_and_testing')

## 1. Annotations and Optional

- `def f(x: int) -> str:` documents what goes in and comes out. Python does not enforce it at runtime; tools like mypy check it.
- `X | None` (or `Optional[X]`) means the value may be missing. Return `None` explicitly in that case.
- Annotations live in `f.__annotations__`.

In [ ]:
def mean_or_none(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)

print(mean_or_none([2, 4]), mean_or_none([]))
print(mean_or_none.__annotations__)

**Exercise 1.** Return the mean, or `None` for an empty list.

In [ ]:
def mean_or_none(values: list[float]) -> float | None:
    if not values:
        return ___
    return sum(values) / len(values)

check(1, (mean_or_none([1, 2, 3]), mean_or_none([])))

**Exercise 2.** Annotations aren't enforced. Predict `double("ab")` for `def double(x: int) -> int: return x * 2`.

In [ ]:
def double(x: int) -> int:
    return x * 2

predicted = ___
check(2, predicted)

**Exercise 3.** Read the declared return type of `area`: annotations store it under the key `"return"`.

In [ ]:
def area(width: float, height: float) -> float:
    return width * height

return_type = area.__annotations__[___]
check(3, return_type.__name__)

## 2. TypedDict and Callable

- `class ProductRecord(TypedDict): sku: str; price: float` describes the keys of a plain dict. At runtime it *is* a dict.
- `Callable[[dict], dict]` means a function taking one dict and returning a dict.
- Passing functions as parameters keeps code reusable: `apply_transform(records, transform)`.

In [ ]:
from typing import TypedDict, Callable

class ProductRecord(TypedDict):
    sku: str
    price: float

record: ProductRecord = {"sku": "A1", "price": 9.5}

def apply_transform(records: list[dict], transform: Callable[[dict], dict]) -> list[dict]:
    return [transform(r) for r in records]

print(type(record).__name__, apply_transform([record], lambda r: {**r, "price": r["price"] * 2}))

**Exercise 4.** A `TypedDict` value is a plain dict at runtime. What is `type(record).__name__`?

In [ ]:
from typing import TypedDict

class User(TypedDict):
    name: str

record: User = {"name": "Ana"}
predicted = ___
check(4, predicted)

**Exercise 5.** `__annotations__` holds the declared keys in order. Predict `keys`.

In [ ]:
from typing import TypedDict

class ProductRecord(TypedDict):
    sku: str
    name: str
    price: float

keys = list(ProductRecord.__annotations__)
predicted = ___
check(5, predicted)

**Exercise 6.** Apply the transform to every record.

In [ ]:
def apply_transform(records, transform):
    return [___ for r in records]

rows = [{"n": 1}, {"n": 2}]
check(6, apply_transform(rows, lambda r: {"n": r["n"] + 10}))

## 3. Protocols and runtime contracts

- A `Protocol` lists the methods a type needs. Any class with them fits, no inheritance required (duck typing that tools can check).
- Hints don't validate data. Check real rules yourself and raise `ValueError` with a clear message.
- Validate in `__post_init__` for dataclasses, or in `__init__`.

In [ ]:
from dataclasses import dataclass
from typing import Protocol

class HasArea(Protocol):
    def area(self) -> float: ...

class Square:
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2

def total_area(shapes: list[HasArea]) -> float:
    return sum(s.area() for s in shapes)

@dataclass
class Settings:
    timeout: float
    def __post_init__(self):
        if self.timeout <= 0:
            raise ValueError("timeout must be > 0")

print(total_area([Square(2), Square(3)]))
try:
    Settings(timeout=0)
except ValueError as error:
    print(error)

**Exercise 7.** `Circle` has an `area` method but doesn't inherit anything. Does `total_area` accept it?

In [ ]:
class Circle:
    def __init__(self, r): self.r = r
    def area(self): return 3 * self.r ** 2

def total_area(shapes):
    return sum(s.area() for s in shapes)

predicted = ___   # total_area([Circle(1), Circle(2)])
check(7, predicted)

**Exercise 8.** Reject a negative `retries` with `ValueError`.

In [ ]:
from dataclasses import dataclass

@dataclass
class Settings:
    retries: int
    def __post_init__(self):
        if ___:
            raise ValueError("retries must be >= 0")

try:
    Settings(retries=-1)
    outcome = "accepted"
except ValueError:
    outcome = "rejected"
check(8, (outcome, Settings(retries=0).retries))

## Challenges

Real functions, graded by tests. Fill every `___`, run the function cell, then run its check.

### 1. `mean_or_none`

The mean, or `None` for an empty list.

In [ ]:
def mean_or_none(values: list[float]) -> Optional[float]:
    """Return the arithmetic mean, or None for an empty list."""
    return ___

In [ ]:
check('mean_or_none')

### 2. `ProductRecord`

A `TypedDict` with `sku`, `name` and `price`.

In [ ]:
class ProductRecord(TypedDict):
    """Shape of one product from the API: sku (str), name (str), price (float)."""
    sku: str
    name: str
    price: float

In [ ]:
check('ProductRecord')

### 3. `apply_transform`

Apply a transform function to every record.

In [ ]:
def apply_transform(records: Iterable[dict], transform: Callable[[dict], dict]) -> list[dict]:
    """Apply `transform` to every record and return the new list."""
    return ___

In [ ]:
check('apply_transform')

### 4. `Settings`

Validate `timeout > 0` and `retries >= 0`.

In [ ]:
class Settings:
    """Client settings. timeout must be > 0 and retries >= 0 (ValueError otherwise)."""
    timeout: float = 5.0
    retries: int = 3

    def __post_init__(self):
        if ___:
            raise ValueError("timeout must be positive")
        if ___:
            raise ValueError("retries cannot be negative")

In [ ]:
check('Settings')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Do type hints stop wrong types at runtime?

   - **a)** No, they are checked by tools, not enforced by Python
   - **b)** Only for int
   - **c)** Yes, always

**2.** A Protocol describes…

   - **a)** A required base class
   - **b)** A network protocol
   - **c)** The methods an object must have (structural typing)

**3.** Optional[int] means…

   - **a)** Any type
   - **b)** An int with a default
   - **c)** int or None

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/10', {1: '83f2c72b3568', 2: '416f48af5d4c', 3: 'b7f3ef96236d'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [11 — Logging, Configuration, and Error Boundaries](11_logging_configuration_and_errors.ipynb)